In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from collections import Counter
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as t
from math import sqrt
import torch.nn as nn
import torch.optim as opt
import torch.nn.functional as F

In [4]:
all_pairs=[]
for u in range(1, 79):
    df=pd.read_excel(f"Dataset/behaviour_biometrics_dataset/feature_kmt_dataset/feature_kmt_xlsx/feature_kmt_user_{u:04d}.xlsx")
    labels=df["label"]
    data=df[["dwell_avg", "flight_avg", "traj_avg"]]
    X=data.to_numpy()
    x_list=X.tolist()
    x_valid=[]
    x_invalid=[]
    positive_pairs=[]
    negative_pairs=[]

    for i in range(10):
        x_valid.append(x_list[i])
    for i in range(10, 20):
        x_invalid.append(x_list[i])
    for i in range(10):
        for j in range(i+1, 10):
            positive_pairs.append([x_valid[i], x_valid[j]])
    for i in range(10):
        for j in range(10):
            negative_pairs.append([x_valid[i], x_invalid[j]])
    pairs=[(a,b,1) for (a,b) in positive_pairs]+[(a,b,0) for (a,b) in negative_pairs]      
    all_pairs.extend(pairs)

In [9]:
import random 
random.seed(42)
random.shuffle(all_pairs)

In [12]:
labels=[p[2] for p in all_pairs]
train_pairs, val_pairs = train_test_split(all_pairs, test_size=0.20, stratify=labels, random_state=42)


In [17]:
dwell_sum = 0
flight_sum = 0
traj_sum = 0

for (x1, x2, x3) in train_pairs:
    [d1,f1,t1]=x1
    [d2,f2,t2]=x2
    dwell_sum=dwell_sum+d1+d2
    flight_sum=flight_sum+f1+f2
    traj_sum=traj_sum+t1+t2
dwell_mean=dwell_sum/(2*len(train_pairs))
flight_mean=flight_sum/(2*len(train_pairs))
traj_mean=traj_sum/(2*len(train_pairs))



In [20]:
dwell_sum_sq = 0
flight_sum_sq = 0
traj_sum_sq = 0

for (x1, x2, x3) in train_pairs:
    [d1,f1,t1]=x1
    [d2,f2,t2]=x2
    dwell_sum_sq=dwell_sum_sq+d1**2+d2**2
    flight_sum_sq=flight_sum_sq+f1**2+f2**2
    traj_sum_sq=traj_sum_sq+t1**2+t2**2
dwell_mean_sq=(dwell_sum_sq/(2*len(train_pairs)))
flight_mean_sq=(flight_sum_sq/(2*len(train_pairs)))
traj_mean_sq=(traj_sum_sq/(2*len(train_pairs)))
dwell_std=sqrt(dwell_mean_sq-dwell_mean**2)
flight_std=sqrt(flight_mean_sq-flight_mean**2)
traj_std=sqrt(traj_mean_sq-traj_mean**2)


In [22]:
for i in range(len(train_pairs)):
    
    (x1, x2, x3)=train_pairs[i]
    [d1,f1,t1]=x1
    scaled_x1=[(d1-dwell_mean)/dwell_std, (f1-flight_mean)/flight_std, (t1-traj_mean)/traj_std]
    [d2,f2,t2]=x2
    scaled_x2=[(d2-dwell_mean)/dwell_std, (f2-flight_mean)/flight_std, (t2-traj_mean)/traj_std]
    train_pairs[i] =[scaled_x1, scaled_x2, x3]
for i in range(len(val_pairs)):
    (x1, x2, x3)=val_pairs[i]
    [d1,f1,t1]=x1
    scaled_x1=[(d1-dwell_mean)/dwell_std, (f1-flight_mean)/flight_std, (t1-traj_mean)/traj_std]
    [d2,f2,t2]=x2
    scaled_x2=[(d2-dwell_mean)/dwell_std, (f2-flight_mean)/flight_std, (t2-traj_mean)/traj_std]
    val_pairs[i] =[scaled_x1, scaled_x2, x3]



In [24]:
class Embedder(nn.Module):
    def __init__(self):
        super(Embedder, self).__init__()
        self.hlayer1=nn.Linear(3,64)
        self.relu1=nn.ReLU()
        self.hlayer2=nn.Linear(64,32)
        self.relu2=nn.ReLU()
        self.flayer=nn.Linear(32,8)
    def forward(self,x):
        x=self.flayer(self.relu2(self.hlayer2(self.relu1(self.hlayer1(x)))))
        return x
    

In [25]:
class Siamese(nn.Module):
    def __init__(self):
        super(Siamese, self).__init__()
        self.embed=Embedder()
    def forward(self, x1, x2):
        return self.embed(x1), self.embed(x2)
        

In [26]:
class contraloss(nn.Module):
    def __init__(self, m):
        super(contraloss, self).__init__()
        self.m=m
    def forward(self, emb1, emb2, label):
        dist=torch.norm(emb1-emb2, p=2, dim=1)
        
        loss_pos=(dist**2)/2
        loss_neg=(F.relu(self.m-dist))**2/2 ##contrastive loss if positive pair diff and neg pair is max(0, m-d)^2/2 and m is margin common values 1,2 ,3 im just tryin out 1
        loss=label*loss_pos+(1-label)*loss_neg
        return loss.mean(), dist.detach()

###
def get_batches(pairs, batch_size):
        random.shuffle(pairs)
        for i in range(0, len(pairs), batch_size):
            batch=pairs[i:i+batch_size]
            x1=[]
            x2=[]
            y=[]
    
            for a, b, c in batch:
                x1.append(a)
                x2.append(b)
                y.append(c)
    
            x1=torch.tensor(x1,dtype=torch.float32)
            x2=torch.tensor(x2,dtype=torch.float32)
            y =torch.tensor(y,dtype=torch.float32)
    
            yield x1, x2, y
###

        

In [27]:
model=Siamese()
criterion= contraloss(m=1)
optimizer=torch.optim.AdamW(model.parameters(), lr=0.001)
for epoch in range(20):
    model.train()

    r_loss=0
    for (x1, x2, x3) in get_batches(train_pairs, 16):
        optimizer.zero_grad()

        emb1, emb2=model(x1, x2)
        loss, dist=criterion(emb1, emb2, x3)
        loss.backward()
        optimizer.step()
        r_loss=r_loss+loss.item()

    print(f"Epoch {epoch} Loss={r_loss}")

Epoch 0 Loss=46.472776083275676
Epoch 1 Loss=40.48702207300812
Epoch 2 Loss=39.28255611099303
Epoch 3 Loss=38.19689347036183
Epoch 4 Loss=37.6780583653599
Epoch 5 Loss=37.12917450815439
Epoch 6 Loss=36.59326444659382
Epoch 7 Loss=36.23232282884419
Epoch 8 Loss=35.73400348331779
Epoch 9 Loss=35.52906403597444
Epoch 10 Loss=35.089180373586714
Epoch 11 Loss=34.79254654701799
Epoch 12 Loss=34.520619054324925
Epoch 13 Loss=34.300822905264795
Epoch 14 Loss=34.12098418548703
Epoch 15 Loss=33.77440627478063
Epoch 16 Loss=33.64490599464625
Epoch 17 Loss=33.233141417615116
Epoch 18 Loss=33.22631787415594
Epoch 19 Loss=32.98752558417618


In [28]:
model.eval()
dist_list=[]
labels=[]
for (x1, x2, x3) in get_batches(val_pairs, 32):
    emb1, emb2=model(x1, x2)
    dist=torch.norm(emb1-emb2, p=2, dim=1)
    dist_list.extend(dist.tolist())  ##append wud add full list as one element but extend adds element wise
    labels.extend(x3.tolist())
positive_dist=[d for d, l in zip(dist_list, labels) if l==1]
negative_dist=[d for d,l in zip(dist_list, labels) if l==0]
mean1=np.mean(positive_dist)
mean2=np.mean(negative_dist)
simple_threshold=(mean1+mean2)/2
    

In [38]:
#train on a user 5 samples of genuine then test on remaining 15

from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix

net_true=[]
net_pred=[]
net_scores=[] 

model.eval()
for u in range(80, 89):

    filename = f"Dataset/behaviour_biometrics_dataset/feature_kmt_dataset/feature_kmt_xlsx/feature_kmt_user_{u:04d}.xlsx"
    df = pd.read_excel(filename)
    data = df[["dwell_avg", "flight_avg", "traj_avg"]].to_numpy()
    data = (data-means)/stds
    genuine_all=data[0:10] 
    imposter_all=data[10:20] 
        
    support_data=genuine_all[0:5]
    test_genuine=genuine_all[5:10]
    test_imposter=imposter_all
    with torch.no_grad():
        support_tensor=torch.tensor(support_data, dtype=torch.float32)
        support_embs=model.embed(support_tensor)
        user_vector=torch.mean(support_embs, dim=0)
    user_true=[]
    user_pred=[]
    user_dists=[]
    with torch.no_grad():
        gtest_tensor=torch.tensor(test_genuine, dtype=torch.float32) 
        gtest_embs=model.embed(gtest_tensor)
        dists=torch.norm(gtest_embs-user_vector, p=2, dim=1).tolist()
        
        user_true.extend([1]*len(dists))
        user_dists.extend(dists)
        user_pred.extend([1 if d<simple_threshold else 0 for d in dists])

    with torch.no_grad():
        itest_tensor=torch.tensor(test_imposter, dtype=torch.float32)
        itest_embs=model.embed(itest_tensor)
        dists=torch.norm(itest_embs-user_vector, p=2, dim=1).tolist()
        
        user_true.extend([0]*len(dists))
        user_dists.extend(dists)
        user_pred.extend([1 if d<simple_threshold else 0 for d in dists])
    
    acc=accuracy_score(user_true, user_pred)
    f1=f1_score(user_true, user_pred, zero_division=0)
    tn, fp, fn, tp=confusion_matrix(user_true, user_pred, labels=[0,1]).ravel()
    far=fp/(fp+tn)
    print(f"User No.={u}, Real Label={user_true}, Output={user_pred} Accuracy={acc}")

    
##    
    net_true.extend(user_true)
    net_pred.extend(user_pred)
    # For AUC, we use Negative Distance (Higher score = Match)
    net_scores.extend([-d for d in user_dists]) 
###



final_acc=accuracy_score(net_true, net_pred)
final_f1=f1_score(net_true, net_pred)
final_auc=roc_auc_score(net_true, net_scores)
tn, fp, fn, tp=confusion_matrix(net_true, net_pred).ravel()
final_far=fp/(fp+tn)
print(f"Net Accuracy={final_acc}")
print(f"Net F1 Score:{final_f1}")
print(f"Net ROC AUC:{final_auc}")
print(f"Net FAR:{final_far}")




User No.=80, Real Label=[1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], Output=[1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0] Accuracy=0.7333333333333333
User No.=81, Real Label=[1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], Output=[1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 0] Accuracy=0.7333333333333333
User No.=82, Real Label=[1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], Output=[1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0] Accuracy=0.8
User No.=83, Real Label=[1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], Output=[1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1] Accuracy=0.9333333333333333
User No.=84, Real Label=[1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], Output=[1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1] Accuracy=0.8
User No.=85, Real Label=[1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], Output=[1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0] Accuracy=0.7333333333333333
User No.=86, Real Label=[1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], Output=[1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 